# NodalMLP probe decoder — hparam importance & parallel coordinates

Reconstructs Optuna studies from existing W&B tune-trial runs for the four probe-decoder/encoder variants of `NodalMLP` (`dot`, `bilinear`, `linear_beta`, and the `spectral`-encoder pair) and computes hparam importance via fANOVA.

**Important: SINCE filter.** `NodalMLP` was refactored to a new `encoder_type`/`embedding_dim` schema; pre-refactor tune trials carry stale hparam keys (`use_encoder`, `use_sc_pca`, `sc_pca_dim`, `hidden_dim`) that no longer exist in the current YAMLs and will be skipped on schema mismatch. Set `SINCE` below to the timestamp at/after which you re-launched the new YAMLs.

Sections:
1. Pull tune-trial runs per variant (with `since` filter), replay into Optuna studies, print importance tables.
2. Parallel-coordinates plot of (hparams → val_demeaned_r) per variant.
3. Side-by-side variant comparison: tune-trial val distributions + best-trial test metrics.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = '/scratch/asr655/neuroinformatics/Conn2Conn'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from results.scripts.optuna_importance import (
    importance_for_variant,
    fetch_tune_trial_runs,
    filter_by_config_match,
)
from results.results_scraper import fetch_best_trial_runs

# Cutoff for run createdAt — set to the moment the new YAMLs were re-launched
# so pre-schema-refactor runs are excluded.
SINCE = '2026-04-27'

# Probe variants we want to compare. (config_stem, decoder_type label, encoder_type filter or None).
# - The first three share decoder_type-as-label semantics.
# - 'spectral' is a YAML where decoder_type is itself searched ({dot, diag_bilinear}); we filter
#   on encoder_type=spectral instead and let decoder_type be the searched axis.
VARIANTS = [
    ('NodalMLP_dot',         'dot',          {'decoder_type': 'dot'}),
    ('NodalMLP_bilinear',    'bilinear',     {'decoder_type': 'bilinear'}),
    ('NodalMLP_linear_beta', 'linear_beta',  {'decoder_type': 'linear_beta'}),
    ('NodalMLP_spectral',    'spectral',     {'encoder_type': 'spectral'}),
]
MODEL_CLASS = 'NodalMLP'
METRIC = 'val_demeaned_r'
DIRECTION = 'maximize'

## 1. Importance per variant

For each variant: fetch `NodalMLP` tune trials created after `SINCE`, filter by config keys, reconstruct trials against the variant's YAML search space, compute fANOVA importance. The script prints per-key skip diagnostics — if everything is skipped because of a missing param, it usually means `SINCE` is too early.

In [ ]:
results = {}
for stem, label, match in VARIANTS:
    print(f'\n===  {label}  (config: {stem})  match={match}  ===')
    res = importance_for_variant(
        model_class_name=MODEL_CLASS,
        config_stem=stem,
        match=match,
        metric_key=METRIC,
        direction=DIRECTION,
        since=SINCE,
        verbose=True,
    )
    results[label] = res
    print(f'  W&B runs fetched (since {SINCE}) : {res["n_runs_fetched"]}')
    print(f'  matched after filter             : {res["diagnostics"].get("n_after_match")}')
    print(f'  trials replayed                  : {res["n_trials_reconstructed"]}')
    if not res['importance']:
        print('  (no valid trials — see diagnostic skip reasons above)')
        continue
    print(f'  best {METRIC:<22} = {res["best_value"]:.5f}')
    print(f'  best params      : {res["best_params"]}')
    print('  fANOVA importance:')
    for k, v in res['importance'].items():
        print(f'    {k:28s}  {v:.4f}')

In [ ]:
# Compact importance table: rows = hparams, cols = variants.
imp_df = pd.DataFrame({label: res['importance'] for label, res in results.items() if res['importance']})
imp_df = imp_df.fillna(np.nan).sort_index()
imp_df

In [ ]:
# Heatmap of importance across variants.
if not imp_df.empty:
    fig, ax = plt.subplots(figsize=(2.2 * len(imp_df.columns) + 1.5, 0.35 * len(imp_df) + 1.0))
    im = ax.imshow(imp_df.values, aspect='auto', cmap='viridis', vmin=0, vmax=max(0.5, np.nanmax(imp_df.values)))
    ax.set_xticks(range(len(imp_df.columns)))
    ax.set_xticklabels(imp_df.columns, rotation=20, ha='right')
    ax.set_yticks(range(len(imp_df.index)))
    ax.set_yticklabels(imp_df.index)
    for i in range(imp_df.shape[0]):
        for j in range(imp_df.shape[1]):
            v = imp_df.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='white' if v > 0.25 else 'black', fontsize=8)
    ax.set_title('fANOVA hparam importance (per probe variant)')
    fig.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout()
    plt.show()
else:
    print('No variants produced valid trials. Check the diagnostic output above —\n'
          'usually means SINCE is set before your most recent retrigger.')

## 2. Parallel-coordinates plot per variant

Each line is a tune trial; axes are hparams (categoricals on integer ticks); color encodes `val_demeaned_r`. Use this to spot 'good slices' of the search space and to sanity-check the importance ranking.

In [ ]:
def variant_trials_df(label, results=results):
    res = results.get(label)
    if not res or not res['study'].trials:
        return pd.DataFrame()
    rows = []
    for t in res['study'].trials:
        row = dict(t.params)
        row[METRIC] = t.value
        row['wandb_run_id'] = t.user_attrs.get('wandb_run_id')
        rows.append(row)
    return pd.DataFrame(rows)

trials_dfs = {label: variant_trials_df(label) for _, label, _ in VARIANTS}
for label, df in trials_dfs.items():
    print(f'{label}: {len(df)} trials')
next((df for df in trials_dfs.values() if not df.empty), pd.DataFrame()).head()

In [ ]:
def plot_parallel_coords(df, label, metric=METRIC):
    if df.empty:
        print(f'{label}: no trials, skipping plot')
        return None
    import plotly.graph_objects as go

    dims = []
    for col in df.columns:
        if col in (metric, 'wandb_run_id'):
            continue
        s = df[col]
        if s.dtype.kind in 'OUSb' or s.apply(lambda x: isinstance(x, (str, bool, tuple))).any():
            cats = list(pd.unique(s.astype(str)))
            cats_sorted = sorted(cats, key=lambda x: (len(x), x))
            mapping = {c: i for i, c in enumerate(cats_sorted)}
            dims.append(dict(
                label=col,
                values=s.astype(str).map(mapping),
                tickvals=list(mapping.values()),
                ticktext=cats_sorted,
            ))
        else:
            dims.append(dict(label=col, values=s.astype(float)))
    dims.append(dict(label=metric, values=df[metric].astype(float)))

    fig = go.Figure(data=go.Parcoords(
        line=dict(color=df[metric], colorscale='Viridis', showscale=True,
                  cmin=df[metric].min(), cmax=df[metric].max(),
                  colorbar=dict(title=metric)),
        dimensions=dims,
    ))
    fig.update_layout(title=f'{label} — parallel coords ({len(df)} trials)', height=420)
    return fig

for label, df in trials_dfs.items():
    fig = plot_parallel_coords(df, label)
    if fig is not None:
        fig.show()

## 3. Side-by-side variant comparison

Tune-trial `val_demeaned_r` distribution per variant, plus best-trial (prod) test metrics aggregated by decoder×seed. The first plot shows search-space coverage; the table shows the chosen-config retrained on each seed and evaluated on the held-out test split.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
data, labels = [], []
for label, df in trials_dfs.items():
    if not df.empty:
        data.append(df[METRIC].values)
        labels.append(f'{label}\n(n={len(df)})')
if data:
    ax.boxplot(data, labels=labels, showfliers=True)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_ylabel(METRIC)
    ax.set_title('Tune-trial val metric distribution per probe variant')
    plt.tight_layout()
    plt.show()
else:
    print('No trials to plot.')

In [ ]:
# Best-trial (prod) summary per variant, restricted to runs created after SINCE.
all_best = fetch_best_trial_runs(MODEL_CLASS)
if SINCE:
    cutoff = pd.Timestamp(SINCE).tz_localize('UTC')
    all_best = [r for r in all_best if pd.Timestamp(r.created_at).tz_convert('UTC') >= cutoff]
print(f'best-trial runs after {SINCE}: {len(all_best)}')

rows = []
for stem, label, match in VARIANTS:
    runs_v = filter_by_config_match(all_best, match)
    for r in runs_v:
        s = dict(r.summary._json_dict if hasattr(r.summary, '_json_dict') else r.summary)
        cfg = dict(r.config)
        seed = (cfg.get('data') or {}).get('shuffle_seed') if isinstance(cfg.get('data'), dict) else cfg.get('shuffle_seed')
        rows.append(dict(
            variant=label,
            seed=seed,
            val_demeaned_r=s.get('val_demeaned_r'),
            test_demeaned_pearson=s.get('eval_test/demeaned_pearson'),
            test_pearson=s.get('eval_test/pearson_r'),
            test_mse=s.get('eval_test/mse'),
            run_name=r.name,
        ))
best_df = pd.DataFrame(rows).sort_values(['variant', 'seed']).reset_index(drop=True)
best_df

In [ ]:
# Mean ± std test metric per variant, across seeds.
if not best_df.empty:
    summary = (best_df
        .groupby('variant')[['val_demeaned_r', 'test_demeaned_pearson', 'test_pearson', 'test_mse']]
        .agg(['mean', 'std', 'count']))
    print(summary)
else:
    print('No best-trial runs found after SINCE.')